### Part 1: TF-IDF & BM25 & Weighted Scoring

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/Shareddrives/RIAW/FINAL_PROJECT/DELIVERABLE 1/

In [2]:
import csv
import re
import csv
import re
import numpy as np
import math
import pandas as pd

from collections import defaultdict
from array import array
from typing import Dict, List, Tuple
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

In [3]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
def build_terms_query(text: str) -> List[str]:
    """Tokenize and normalize text into a list of lowercase terms.
    - Keeps only alphanumeric characters
    - Splits on non-alphanumerics
    """
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    if not text:
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return tokens

In [5]:
def get_document(csv_path: str):
  with open(csv_path, mode="r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        documentos = list(reader)

  return pd.DataFrame(documentos)

In [6]:
def custom_score(document, query_tokens):

    # Extraemos listas de palabras del documento
    title_tokens = document["title"]
    desc_tokens = document["description"]
    brand_tokens = document["brand"]

    # 1. Matches
    title_match = sum(1 for w in query_tokens if w in title_tokens)
    desc_match = sum(1 for w in query_tokens if w in desc_tokens)
    brand_match = 1 if any(w in brand_tokens for w in query_tokens) else 0

    # 2. Weighted score
    score = (
        0.6 * title_match +
        0.3 * desc_match +
        0.1 * brand_match
    )

    return score

In [7]:
def print_ranked_results(query_text, ranking, df, top_k=10):
    print(f"\n🔍 Query: {query_text}\n")

    for i, (pid, score) in enumerate(ranking[:top_k], start=1):

        # buscar documento en el DataFrame
        doc = df[df["pid"] == pid].iloc[0]

        # reconstruir título a partir de tokens
        title = doc["title"]

        print(f"{i}. [{score:.3f}] {title} (id={pid})")


In [8]:
csv_path = "../../data/productos_preprocesados.csv"
df = get_document(csv_path)

In [9]:
# Build a single text field per document (title + description + brand)
def build_full_text(row):
    return f"{row['title']} {row['description']} {row['brand']}"

df["full_text"] = df.apply(build_full_text, axis=1)

# Tokenized version of each document using the same preprocessing as queries
df["tokens"] = df["full_text"].apply(build_terms_query)


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF vectoriser that uses the same tokeniser as build_terms_query
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=build_terms_query,   # use preprocessing
    preprocessor=lambda x: x,      # x is already string
    token_pattern=None             # disable internal token pattern
)

# Fit TF-IDF on the full document texts
tfidf_matrix = tfidf_vectorizer.fit_transform(df["full_text"])


In [11]:
def filter_conjunctive(query_tokens):
    """
    Returns the indices of documents that contain ALL query tokens.
    We use the precomputed df["tokens"] (list of tokens per doc).
    """
    matching_indices = []

    for idx, doc_tokens in df["tokens"].items():
        if all(t in doc_tokens for t in query_tokens):
            matching_indices.append(idx)

    return matching_indices


In [12]:
def rank_tfidf(query_text, top_k=20):
    """
    Ranking with TF-IDF + cosine similarity, restricted to
    documents that satisfy the conjunctive (AND) query.
    Returns a list of (pid, score).
    """
    # Preprocess query
    query_tokens = build_terms_query(query_text)

    # Conjunctive filter
    matching_indices = filter_conjunctive(query_tokens)

    if not matching_indices:
        return []

    # Represent query as TF-IDF vector
    query_vec = tfidf_vectorizer.transform([" ".join(query_tokens)])

    # Cosine similarity with matching documents
    sims = cosine_similarity(query_vec, tfidf_matrix[matching_indices]).flatten()

    # Collect (pid, score)
    scores = []
    for local_i, doc_idx in enumerate(matching_indices):
        pid = df.iloc[doc_idx]["pid"]
        scores.append((pid, sims[local_i]))

    # Sort by similarity descending
    scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores_sorted[:top_k]


In [13]:
class BM25:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        """
        corpus_tokens: list of lists of tokens, i.e. df["tokens"].tolist()
        """
        self.corpus = corpus_tokens
        self.N = len(self.corpus)
        self.k1 = k1
        self.b = b

        self.doc_len = [len(doc) for doc in self.corpus]
        self.avgdl = sum(self.doc_len) / self.N

        # Document frequencies and IDF
        self.df = defaultdict(int)
        for doc in self.corpus:
            for w in set(doc):
                self.df[w] += 1

        self.idf = {}
        for w, df in self.df.items():
            # classic BM25 idf
            self.idf[w] = math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, doc_index, query_tokens):
        """
        BM25 score between a document and a query.
        """
        score = 0.0
        doc = self.corpus[doc_index]
        if not doc:
            return 0.0

        # term frequencies in the document
        freqs = defaultdict(int)
        for w in doc:
            freqs[w] += 1

        dl = self.doc_len[doc_index]

        for w in query_tokens:
            if w not in freqs:
                continue

            tf = freqs[w]
            idf = self.idf.get(w, 0)

            num = tf * (self.k1 + 1)
            den = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)

            score += idf * num / den

        return score

# Initialize BM25 with your document tokens
bm25 = BM25(df["tokens"].tolist())


In [14]:
def rank_bm25(query_text, top_k=20):
    """
    Ranking with BM25, restricted to documents that satisfy
    the conjunctive (AND) query.
    Returns a list of (pid, score).
    """
    query_tokens = build_terms_query(query_text)

    # Conjunctive filter
    matching_indices = filter_conjunctive(query_tokens)

    if not matching_indices:
        return []

    scores = []
    for doc_idx in matching_indices:
        s = bm25.score(doc_idx, query_tokens)
        pid = df.iloc[doc_idx]["pid"]
        scores.append((pid, s))

    scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores_sorted[:top_k]


In [ ]:
resultados_custom = {}
resultados_tfidf = {}
resultados_bm25 = {}

test_queries = [
    "cotton multicolor track pant",
    "women black track pant pockets",
    "men cotton blue pant",
    "side pocket cotton pant",
    "slim cotton black pant",
]

for i, query in enumerate(test_queries, start=1):
    query_tokens = build_terms_query(query)

    # 1) Custom score
    scores_custom = []
    for idx, doc in df.iterrows():
        s = custom_score(doc, query_tokens)
        scores_custom.append((doc["pid"], s))

    scores_custom_sorted = sorted(scores_custom, key=lambda x: x[1], reverse=True)
    resultados_custom[f"query_{i}"] = scores_custom_sorted

    # 2) TF-IDF + cosine
    resultados_tfidf[f"query_{i}"] = rank_tfidf(query, top_k=20)

    # 3) BM25
    resultados_bm25[f"query_{i}"] = rank_bm25(query, top_k=20)


In [16]:
for i, query in enumerate(test_queries, start=1):
    print("\n" + "="*80)
    print(f"QUERY {i}: {query}")
    print("="*80)

    print("\n--- Custom Score ---")
    print_ranked_results(query, resultados_custom[f"query_{i}"], df, top_k=20)

    print("\n--- TF-IDF + Cosine Similarity ---")
    print_ranked_results(query, resultados_tfidf[f"query_{i}"], df, top_k=20)

    print("\n--- BM25 ---")
    print_ranked_results(query, resultados_bm25[f"query_{i}"], df, top_k=20)



QUERY 1: cotton multicolor track pant

--- Custom Score ---

🔍 Query: cotton multicolor track pant

1. [2.700] solid women multicolor track pant (id=TKPFCZ9EA7H5FYZH)
2. [2.700] solid men multicolor track pant (id=TKPFCZ9EHFCY5Z4Y)
3. [2.700] solid women multicolor track pant (id=TKPFCZ9ESZZ7YWEF)
4. [2.700] solid women multicolor track pant (id=TKPFCZ9EFK9DNWDA)
5. [2.700] solid women multicolor track pant (id=TKPFCZ9EHCNAPKPU)
6. [2.700] color block women multicolor track pant (id=TKPFCZ9EGGYENTZS)
7. [2.700] solid women multicolor track pant (id=TKPFCZ9EZDPZR5AH)
8. [2.700] solid women multicolor track pant (id=TKPFDHDGEYWM99XG)
9. [2.700] solid men multicolor track pant (id=TKPFDHDHGWXCFXPC)
10. [2.700] camouflag women multicolor track pant (id=TKPEJG8DEG5FA8VK)
11. [2.700] print men multicolor track pant (id=TKPFGJTAAT58GPA3)
12. [2.400] solid women multicolor track pant (id=TKPFK52C7YUQAABT)
13. [2.400] solid men multicolor track pant (id=TKPFK52CRCHV9HQ7)
14. [2.400] solid wome